In [288]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from statsmodels.tsa.stattools import pacf
import numpy as np

In [289]:
import os
import sys
current_dir = os.path.dirname(os.path.realpath(__name__))
parent_dir = os.path.dirname(current_dir)
print(parent_dir)
sys.path.append(parent_dir)
from src.data.data_loader import DataLoader
dl = DataLoader()
dl.prepare_df()
prepared_df = dl.prepared_df
dl.train_test_split()

C:\Users\jonah.eisen\dsi\global-stock-index-ml-classification


In [290]:
train_pacf = pacf(dl.y_train.rolling(window=7).mean().dropna(), method='ywm')
alpha = 0.1
ar_terms = [i for i,val in enumerate(np.abs(train_pacf) > alpha) if val]
ar_terms = ar_terms[1:]
ar_terms

[1, 2, 3, 4, 6, 7, 8, 9, 11, 14, 15, 16, 21, 22, 23, 29]

In [291]:
prepared_df.head()

,Last Close,Close,Last Adj Close,Volume,LowProportion,HighProportion,Delta Close
Date,,,,,,,
2003-01-10,5210.040039,5209.209961,5210.339844,1.477200e+09,0.991511,1.004248,-0.830078
2003-01-13,5209.799805,5233.660156,5209.799805,1.371500e+09,0.995975,1.006883,23.860351
2003-01-14,5209.209961,5171.450195,5209.209961,1.355600e+09,0.996499,1.004696,-37.759766
2003-01-15,5233.729980,5165.339844,5233.660156,1.385200e+09,0.986618,1.000290,-68.390136
2003-01-16,5171.450195,5108.509766,5171.450195,1.534600e+09,0.996568,1.006555,-62.940429


In [292]:
p = max(ar_terms)
lag_dict = {}
for j in ar_terms:
    lag_dict[f'y_(t-{j})'] = []
for i in range(p,prepared_df.shape[0]):
    for j in ar_terms:
        lag_val = prepared_df['Delta Close'].iloc[i-j]
        lag_dict[f'y_(t-{j})'].append(lag_val) 

In [293]:
cropped_df = prepared_df.iloc[p:,:]
cropped_df.shape

(4598, 7)

In [294]:
cropped_df = cropped_df.assign(**lag_dict)
cropped_df.shape

(4598, 23)

In [295]:
cropped_df.head()

,Last Close,Close,Last Adj Close,Volume,LowProportion,HighProportion,Delta Close,y_(t-1),y_(t-2),y_(t-3),...,y_(t-8),y_(t-9),y_(t-11),y_(t-14),y_(t-15),y_(t-16),y_(t-21),y_(t-22),y_(t-23),y_(t-29)
Date,,,,,,,,,,,,,,,,,,,,,
2003-02-24,4787.169922,4713.379883,4787.169922,1.219200e+09,0.983069,1.000054,-73.790039,-24.629883,23.699707,-74.890136,...,-83.060059,-4.970215,-94.199707,-44.220215,100.359863,2.720215,-82.790039,-20.619629,-145.529786,-0.830078
2003-02-25,4706.220215,4653.979980,4706.220215,1.495500e+09,0.982515,1.002097,-52.240235,-73.790039,-24.629883,23.699707,...,-55.439941,-83.060059,-24.540039,-76.930176,-44.220215,100.359863,-208.700195,-82.790039,-20.619629,23.860351
2003-02-26,4713.370117,4693.529785,4713.379883,1.382300e+09,0.986621,1.000000,-19.840332,-52.240235,-73.790039,-24.629883,...,80.040039,-55.439941,-4.970215,-67.170410,-76.930176,-44.220215,-39.199707,-208.700195,-82.790039,-37.759766
2003-02-27,4654.000000,4716.069824,4653.979980,1.297100e+09,0.998593,1.012544,62.069824,-19.840332,-52.240235,-73.790039,...,152.479980,80.040039,-83.060059,-94.199707,-67.170410,-76.930176,78.979981,-39.199707,-208.700195,-68.390136
2003-02-28,4693.529785,4709.379883,4693.529785,1.311700e+09,1.000000,1.010189,15.850098,62.069824,-19.840332,-52.240235,...,33.700195,152.479980,-55.439941,-24.540039,-94.199707,-67.170410,-56.540039,78.979981,-39.199707,-62.940429


In [296]:
delta_close = cropped_df['Delta Close']
cropped_df.drop(columns=['Last Close', 'Close', 'Delta Close', 'Last Adj Close'],inplace=True)

In [297]:
cropped_df.head()

,Volume,LowProportion,HighProportion,y_(t-1),y_(t-2),y_(t-3),y_(t-4),y_(t-6),y_(t-7),y_(t-8),y_(t-9),y_(t-11),y_(t-14),y_(t-15),y_(t-16),y_(t-21),y_(t-22),y_(t-23),y_(t-29)
Date,,,,,,,,,,,,,,,,,,,
2003-02-24,1.219200e+09,0.983069,1.000054,-24.629883,23.699707,-74.890136,33.700195,80.040039,-55.439941,-83.060059,-4.970215,-94.199707,-44.220215,100.359863,2.720215,-82.790039,-20.619629,-145.529786,-0.830078
2003-02-25,1.495500e+09,0.982515,1.002097,-73.790039,-24.629883,23.699707,-74.890136,152.479980,80.040039,-55.439941,-83.060059,-24.540039,-76.930176,-44.220215,100.359863,-208.700195,-82.790039,-20.619629,23.860351
2003-02-26,1.382300e+09,0.986621,1.000000,-52.240235,-73.790039,-24.629883,23.699707,33.700195,152.479980,80.040039,-55.439941,-4.970215,-67.170410,-76.930176,-44.220215,-39.199707,-208.700195,-82.790039,-37.759766
2003-02-27,1.297100e+09,0.998593,1.012544,-19.840332,-52.240235,-73.790039,-24.629883,-74.890136,33.700195,152.479980,80.040039,-83.060059,-94.199707,-67.170410,-76.930176,78.979981,-39.199707,-208.700195,-68.390136
2003-02-28,1.311700e+09,1.000000,1.010189,62.069824,-19.840332,-52.240235,-73.790039,23.699707,-74.890136,33.700195,152.479980,-55.439941,-24.540039,-94.199707,-67.170410,-56.540039,78.979981,-39.199707,-62.940429


In [298]:
len(lag_dict.keys())

16

In [299]:
val_size = 0.2
X_train, X_test = DataLoader.time_split_2D(cropped_df)
X_train, X_val = DataLoader.time_split_2D(X_train)
y_train, y_test = DataLoader.time_split_1D(delta_close)
y_train, y_val = DataLoader.time_split_1D(y_train)

In [300]:
#X_train  = X_train.rolling(window=7, center=False).mean().dropna()
#y_train  = y_train.rolling(window=7, center=False).mean().dropna()
y_train = DataLoader.quantize_delta_close(y_train)
y_val = DataLoader.quantize_delta_close(y_val)

In [301]:
X_train.head()
y_train.head()

Date
2003-02-24    0
2003-02-25    0
2003-02-26    0
2003-02-27    1
2003-02-28    1
dtype: int64

In [302]:
rf = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
y_prob = rf.predict_proba(X_val)[:,1]



In [303]:
print("Random Forest Accuracy:", rf.score(X_val, y_val))
print(classification_report(y_val, y_pred))
print("ROC-AUC:", roc_auc_score(y_val, y_prob))

Random Forest Accuracy: 0.686141304347826
              precision    recall  f1-score   support

           0       0.70      0.56      0.62       340
           1       0.68      0.80      0.73       396

    accuracy                           0.69       736
   macro avg       0.69      0.68      0.68       736
weighted avg       0.69      0.69      0.68       736

ROC-AUC: 0.7546865715983363
